# Pack 08 — Channels and BER: AWGN, Multipath, Rayleigh/Rician, Doppler, Eb/N0 vs SNR

**Module 05-dsp · Etherstack interview prep · C++ exercises in `code/`, plots and numerical demos here.**

| | |
|---|---|
| Time | ~2.5 h |
| You need | Python 3 + NumPy + Matplotlib; CMake + C++17 |
| Signals | synthetic only (seeded random bits, simulated channels) |
| Etherstack relevance | Every receiver is specified by a sensitivity/BER target in noise **and** in fading; engineers convert between dBm, SNR, Eb/N0 and BER constantly. |

Notation: $E_b$ energy per bit, $E_s$ per symbol, $N_0$ one-sided noise PSD (W/Hz), $R_b$ bit rate, $R_s$ symbol rate, $B$ noise bandwidth, $\gamma_b = E_b/N_0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erfc

rng = np.random.default_rng(2024)

def Q(x):
    x = np.asarray(x, dtype=float)
    return 0.5 * np.vectorize(erfc)(x / np.sqrt(2.0))

def qpsk_symbols(nsym):
    b = rng.integers(0, 2, (nsym, 2))
    return ((1 - 2 * b[:, 0]) + 1j * (1 - 2 * b[:, 1])) / np.sqrt(2.0), b     # Es = 1

def awgn(x, n0):
    """Complex noise CN(0, n0): each of I and Q has variance n0/2."""
    return x + np.sqrt(n0 / 2.0) * (rng.standard_normal(x.shape) + 1j * rng.standard_normal(x.shape))

fig, ax = plt.subplots(1, 3, figsize=(12, 3.8))
s, _ = qpsk_symbols(2000)
for a_, esn0 in zip(ax, (4.0, 10.0, 20.0)):
    r = awgn(s, 10 ** (-esn0 / 10))              # Es = 1  ->  N0 = 1/(Es/N0)
    a_.plot(r.real, r.imag, ".", ms=2); a_.set_title(f"QPSK, Es/N0 = {esn0:.0f} dB")
    a_.set_aspect("equal"); a_.set_xlim(-2, 2); a_.set_ylim(-2, 2); a_.grid(True)
plt.tight_layout(); plt.show()

## 1. AWGN: derive the BPSK / QPSK bit-error rate

**Model.** Received sample (matched-filter output, Pack 07) for BPSK symbol $a=\pm1$ with energy $E_b$:

$$r = a\sqrt{E_b} + n,\qquad n\sim\mathcal N(0,\ N_0/2).$$

(Real noise in one dimension has variance $N_0/2$ — half the one-sided PSD, because $N_0$ is split between I and Q.)

**Error event (a = +1 sent):** $n < -\sqrt{E_b}$.

$$P_b = \int_{\sqrt{E_b}}^{\infty}\frac{1}{\sqrt{\pi N_0}}e^{-x^2/N_0}dx = Q\!\left(\frac{\sqrt{E_b}}{\sqrt{N_0/2}}\right)=\boxed{Q\!\left(\sqrt{\tfrac{2E_b}{N_0}}\right)},\qquad Q(x)=\tfrac12\operatorname{erfc}\!\left(\tfrac{x}{\sqrt2}\right).$$

**QPSK:** the I and Q branches are two independent BPSK streams on orthogonal carriers; each carries one bit with energy $E_b = E_s/2$ and sees independent noise of variance $N_0/2$, so **the BER vs $E_b/N_0$ is identical to BPSK** (the symbol error rate is $\approx 2p$ — about double). QPSK doubles the data rate in the same bandwidth for no BER penalty vs $E_b/N_0$ — which is why it is "free".

Reference points (derived from $Q^{-1}$): BER $10^{-2}$ → $E_b/N_0 = 4.3$ dB; $10^{-3}$ → 6.8 dB; $10^{-5}$ → 9.6 dB.

Simulate and compare. The simulation uses $E_s = 1$ so $E_b = 0.5$ and $N_0 = E_b/(E_b/N_0)$.

In [ ]:
def ber_qpsk(ebn0_db, nsym):
    s, b = qpsk_symbols(nsym)
    n0 = 0.5 / 10 ** (ebn0_db / 10)
    r = awgn(s, n0)
    bh = np.stack([r.real < 0, r.imag < 0], axis=1).astype(int)
    return np.mean(bh != b)

ebn0 = np.arange(0, 10)
sim = np.array([ber_qpsk(x, 500_000) for x in ebn0])
theory = Q(np.sqrt(2 * 10 ** (ebn0 / 10)))
for x, a, t in zip(ebn0, sim, theory):
    print(f"Eb/N0 = {x:2d} dB   simulated {a:.3e}   theory {t:.3e}   ratio {a / t:.3f}")
plt.figure(figsize=(7, 4.2))
plt.semilogy(ebn0, theory, "-", label="Q(sqrt(2Eb/N0))")
plt.semilogy(ebn0, sim, "o", label="QPSK simulation (Gray)")
plt.xlabel("Eb/N0 [dB]"); plt.ylabel("BER"); plt.ylim(1e-6, 1); plt.grid(True, which="both"); plt.legend()
plt.tight_layout(); plt.show()

## 2. Eb/N0 versus SNR — derived

Let the received signal power be $P$ and the noise power in bandwidth $B$ be $N = N_0 B$. Then

* $E_b = P/R_b$ (energy per bit = power × bit time), $E_s = P/R_s$.
* **SNR** $= P/N = P/(N_0B)$.

$$\frac{E_b}{N_0} = \frac{P/R_b}{N_0}= \frac{P}{N_0 B}\cdot\frac{B}{R_b}=\text{SNR}\cdot\frac{B}{R_b},\qquad \frac{E_s}{N_0}=\text{SNR}\cdot\frac{B}{R_s}=\frac{E_b}{N_0}\log_2 M.$$

In dB: $E_b/N_0 = \text{SNR} + 10\log_{10}(B/R_b)$. $E_b/N_0$ is the fair way to compare modulations (it removes the bandwidth/rate choice); SNR is what the receiver's front end measures.

*Worked example (P25-like numbers; VERIFY the standard):* $R_b$ = 9600 bit/s, noise bandwidth $B$ = 12.5 kHz: $10\log_{10}(12500/9600)$ = +1.15 dB, so $E_b/N_0$ = SNR + 1.15 dB. To get $E_b/N_0$ = 9.6 dB you need SNR = 8.45 dB.

*Sample-domain version (what your simulator does).* At $s$ samples/symbol with complex noise variance $\sigma^2$ per sample ($= N_0 f_s$, noise filling the whole sampling bandwidth) and signal power per sample $P_s = E_sR_s$:
SNR$_{\rm sample}$ = $E_sR_s/(N_0f_s)$ = $(E_s/N_0)/s$. At one sample/symbol SNR$_{\rm sample}$ = $E_s/N_0$ = $(E_b/N_0)\cdot\log_2M$ — check it below.

In [ ]:
ebn0_db = 6.0
s, _ = qpsk_symbols(400_000)
n0 = 0.5 / 10 ** (ebn0_db / 10)
noise = awgn(np.zeros_like(s), n0)
snr_meas = 10 * np.log10(np.mean(np.abs(s) ** 2) / np.mean(np.abs(noise) ** 2))
print(f"measured per-sample SNR = {snr_meas:.3f} dB")
print(f"Eb/N0 + 10log10(bits/symbol) = {ebn0_db + 10 * np.log10(2):.3f} dB   (sps = 1)")
print(f"P25-style conversion: 12 dB SNR in 12.5 kHz at 9600 bit/s -> Eb/N0 = {12 + 10 * np.log10(12500 / 9600):.2f} dB")
print(f"thermal noise floor in 12.5 kHz: {-174 + 10 * np.log10(12500):.1f} dBm  (-174 dBm/Hz at 290 K)")

## 3. Multipath, delay spread and flat vs frequency-selective fading

A radio signal reaches the receiver via several paths: $y[n] = \sum_l h_l\, x[n-\ell_l]$. Two paths with relative delay $\Delta\tau$ and amplitude ratio $\rho$ give a frequency response $|1 + \rho e^{-j2\pi f\Delta\tau}|$ with notches every $1/\Delta\tau$ Hz (the picture below).

* **Delay spread** $\sigma_\tau$ (rms): the time-spread of the channel. **Coherence bandwidth** $B_c\approx 1/(5\sigma_\tau)$ (rule of thumb, 0.5 correlation; **VERIFY** constant — other definitions use 1/(50σ) for 0.9 correlation).
* If the signal bandwidth $W \ll B_c$: **flat fading** — one complex gain multiplies the whole signal (this is the Rayleigh/Rician case below). If $W \gtrsim B_c$: **frequency-selective** — ISI; needs an equaliser/OFDM (Pack 05).
* *Radio context:* a 12.5 kHz channel needs $B_c \gg 12.5$ kHz ⇒ $\sigma_\tau \ll 16\,\mu$s. Typical urban rms delay spreads are of order a microsecond or a few (**VERIFY** for the deployment), so narrowband LMR is almost always **flat-fading**; broadband (LTE-type) systems are not.

In [ ]:
f = np.linspace(-0.5, 0.5, 2000)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for rho in (0.5, 0.9):
    H = np.abs(1 + rho * np.exp(-2j * np.pi * f * 5))     # path 2 delayed by 5 samples
    ax[0].plot(f, 20 * np.log10(H), label=f"rho={rho}")
ax[0].set_xlabel("frequency [cycles/sample]"); ax[0].set_ylabel("|H| [dB]"); ax[0].set_title("two-path channel, delay 5 samples (notch spacing 1/5)"); ax[0].legend(); ax[0].grid(True)
print("rho=0.5 -> peak", round(20 * np.log10(1.5), 2), "dB, notch", round(20 * np.log10(0.5), 2), "dB")
ax[1].plot(f, 20 * np.log10(np.abs(1 + 0.5 * np.exp(-2j * np.pi * f * 0.2))), label="delay 0.2 samples -> nearly flat across |f|<0.2")
ax[1].set_title("short delay spread => flat fading over a narrow signal"); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()

## 4. Rayleigh fading

### 4.1 Why Rayleigh (derivation sketch)
With many scattered paths and no dominant one, the sum $h=\sum_m c_m e^{j\phi_m}$ has I and Q components that are sums of many independent contributions ⇒ by the central limit theorem $h\sim\mathcal{CN}(0,\ 1)$ (unit mean power). The envelope $|h|$ is Rayleigh, and the instantaneous power $|h|^2$ is exponential with mean 1:

$$P(|h|^2<x)=1-e^{-x}.$$

Deep fades are common: $P(|h|^2 < 0.1)=9.5\ \%$ (−10 dB), $P(|h|^2<0.01)=1\ \%$ (−20 dB), $P(|h|^2<0.001)=0.1\ \%$ (−30 dB).

### 4.2 Doppler and Clarke's model
A mobile at speed $v$ sees each path Doppler-shifted by $f_d\cos\alpha$, $f_d = v f_c/c$. Clarke's sum-of-sinusoids:

$$h(t)=\frac1{\sqrt M}\sum_{m=1}^{M}e^{j(2\pi f_d\cos\alpha_m\,t+\phi_m)},\quad \alpha_m,\phi_m\sim U[0,2\pi).$$

Its autocorrelation is $R(\tau)=J_0(2\pi f_d\tau)$ (a Bessel function; first zero at $f_d\tau = 0.383$). **Coherence time** $T_c\approx 0.423/f_d$ (Rappaport; **VERIFY** — alternatives: $1/f_d$, $9/(16\pi f_d)=0.179/f_d$).

*Example:* 100 km/h = 27.8 m/s at 400 MHz: $f_d = 27.8\times4\times10^8/3\times10^8 = 37.1$ Hz ⇒ $T_c\approx 11.4$ ms. A 4800 sym/s system sends 55 symbols per $T_c$: the channel is **slow** relative to symbols but changes within a ~30 ms TDMA slot/long frame.

In [ ]:
def clarke(n, fd, M=64):
    alpha = rng.uniform(0, 2 * np.pi, M); phi = rng.uniform(0, 2 * np.pi, M)
    k = np.arange(n)[:, None]
    return np.exp(1j * (2 * np.pi * fd * np.cos(alpha) * k + phi)).sum(axis=1) / np.sqrt(M)

fd = 0.01                                   # cycles/sample
h = clarke(20000, fd)
print("mean power:", np.mean(np.abs(h) ** 2).round(3), "  P(|h|^2<0.1):", np.mean(np.abs(h) ** 2 < 0.1).round(4), "(theory 0.0952)")

def j0(x):
    th = np.linspace(0, np.pi, 2001); d = th[1] - th[0]
    v = np.cos(np.outer(x, np.sin(th)))
    return d * (v.sum(axis=1) - 0.5 * (v[:, 0] + v[:, -1])) / np.pi

lags = np.arange(0, 301)
R = np.zeros(len(lags))
for _ in range(20):                         # average several realisations
    hh = clarke(4000, fd)
    R += np.array([np.mean(hh[l:] * np.conj(hh[:len(hh) - l])).real for l in lags])
R /= 20
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].plot(20 * np.log10(np.abs(h[:2000]))); ax[0].set_title("fading envelope [dB]"); ax[0].set_xlabel("sample"); ax[0].grid(True)
ax[1].hist(np.abs(h) ** 2, bins=60, density=True); xs = np.linspace(0, 5, 200); ax[1].plot(xs, np.exp(-xs), "r"); ax[1].set_xlim(0, 5); ax[1].set_title("|h|^2 vs exp(-x)")
ax[2].plot(lags * fd, R, label="simulated"); ax[2].plot(lags * fd, j0(2 * np.pi * fd * lags), "--", label="J0(2 pi fd tau)"); ax[2].legend(); ax[2].set_xlabel("fd * tau"); ax[2].grid(True)
plt.tight_layout(); plt.show()

### 4.3 BER in Rayleigh fading

For coherent BPSK in flat Rayleigh fading with ideal channel knowledge, $\gamma = |h|^2 E_b/N_0$ is exponential with mean $\bar\gamma$. Averaging $Q(\sqrt{2\gamma})$ over that density gives (standard result, e.g. Proakis):

$$P_b=\tfrac12\left(1-\sqrt{\frac{\bar\gamma}{1+\bar\gamma}}\right)\ \xrightarrow{\ \bar\gamma\gg1\ }\ \frac1{4\bar\gamma}.$$

**The BER falls only one decade per 10 dB** (slope 1) instead of the waterfall of AWGN — because errors are dominated by rare deep fades. Numerically, $\bar\gamma = 10$ (10 dB): Rayleigh 0.0233 vs AWGN $3.9\times10^{-6}$.

Hence the radio engineer's **fade margin**: for $10^{-3}$ you need 6.8 dB in AWGN but ≈ 24 dB in Rayleigh (+17 dB). Countermeasures: diversity (two antennas ⇒ slope 2), interleaving + FEC (spread a fade over many codewords), equalisation/OFDM, frequency hopping, power control.

In [ ]:
def ber_bpsk_rayleigh(ebn0_db, nbits):
    a = 1 - 2 * rng.integers(0, 2, nbits)
    h = (rng.standard_normal(nbits) + 1j * rng.standard_normal(nbits)) / np.sqrt(2)      # iid CN(0,1): ideal interleaving
    n0 = 1 / 10 ** (ebn0_db / 10)                                                       # Eb = 1
    w = np.sqrt(n0 / 2) * (rng.standard_normal(nbits) + 1j * rng.standard_normal(nbits))
    z = (np.conj(h) * (h * a + w)).real                                                 # coherent combining with known h
    return np.mean((z < 0) != (a < 0))

ebn0 = np.arange(0, 31, 3)
sim = np.array([ber_bpsk_rayleigh(x, 1_000_000) for x in ebn0])
g = 10 ** (ebn0 / 10)
ray = 0.5 * (1 - np.sqrt(g / (1 + g)))
awgn_t = Q(np.sqrt(2 * g))
for x, a_, t in zip(ebn0, sim, ray):
    print(f"{x:2d} dB  sim {a_:.3e}  Rayleigh theory {t:.3e}")
plt.figure(figsize=(7, 4.2))
plt.semilogy(ebn0, awgn_t, label="AWGN"); plt.semilogy(ebn0, ray, label="Rayleigh (theory)"); plt.semilogy(ebn0, sim, "o", label="Rayleigh (simulated)")
plt.ylim(1e-6, 1); plt.xlabel("Eb/N0 [dB]"); plt.ylabel("BER"); plt.grid(True, which="both"); plt.legend(); plt.tight_layout(); plt.show()

## 5. Rician fading

With a dominant line-of-sight (LOS) path plus scatter, $h=\sqrt{\tfrac{K}{K+1}}e^{j\theta_0}+\sqrt{\tfrac1{K+1}}\,\mathcal{CN}(0,1)$ with **K-factor** $K$ = LOS power / scattered power. $K=0$ is Rayleigh; $K\to\infty$ is a pure AWGN channel. Mean power stays 1. The variance of $|h|^2$ is $(1+2K)/(1+K)^2$ (1 for Rayleigh, $\to 0$ as $K\to\infty$), a quick "how deep are the fades" measure.

In [ ]:
def rician(n, K):
    los = np.sqrt(K / (K + 1.0)) * np.exp(1j * rng.uniform(0, 2 * np.pi))
    scat = np.sqrt(1.0 / (K + 1.0)) * (rng.standard_normal(n) + 1j * rng.standard_normal(n)) / np.sqrt(2)
    return los + scat

def ber_bpsk_h(h, ebn0_db):
    n = len(h); a = 1 - 2 * rng.integers(0, 2, n)
    n0 = 1 / 10 ** (ebn0_db / 10)
    w = np.sqrt(n0 / 2) * (rng.standard_normal(n) + 1j * rng.standard_normal(n))
    return np.mean(((np.conj(h) * (h * a + w)).real < 0) != (a < 0))

print(" K (dB)   var|h|^2 (theory)   BER @ 10 dB Eb/N0")
for kdb in (-100, 0, 3, 6, 10, 15):
    K = 10 ** (kdb / 10)
    h = rician(1_000_000, K)
    print(f"{kdb:6d}   {np.var(np.abs(h) ** 2):.3f} ({(1 + 2 * K) / (1 + K) ** 2:.3f})        {ber_bpsk_h(h, 10):.3e}")
print("AWGN limit:", f"{float(Q(np.sqrt(20))):.3e}")

## 6. Higher-order modulation costs Eb/N0 (numerical check of the Pack 05 claim)

Gray 16-QAM: per axis 4-PAM with levels $\pm1,\pm3$ (normalised so $E_s=1$ ⇒ half-spacing $1/\sqrt{10}$). Per-axis symbol error $\approx 1.5\,Q\big(\tfrac{1/\sqrt{10}}{\sqrt{N_0/2}}\big)$, and with $E_s/N_0 = 4E_b/N_0$ the argument becomes $\sqrt{0.8\,E_b/N_0}$; one bit error per symbol error with Gray mapping, two bits per axis:

$$P_b\approx\tfrac34 Q\!\left(\sqrt{0.8\,\gamma_b}\right).$$

For $10^{-3}$: $Q(x)=1.33\times10^{-3}\Rightarrow x = 3.0\Rightarrow \gamma_b = 9/0.8 = 11.25$ (10.5 dB) vs 6.8 dB for QPSK: **+3.7 dB** (≈ 4 dB).

In [ ]:
def ber_16qam(ebn0_db, nsym):
    bits = rng.integers(0, 2, (nsym, 4))
    lev = {(0, 0): -3, (0, 1): -1, (1, 1): 1, (1, 0): 3}                    # Gray per axis
    I = np.array([lev[(a, b)] for a, b in bits[:, :2]]) / np.sqrt(10)
    Qa = np.array([lev[(a, b)] for a, b in bits[:, 2:]]) / np.sqrt(10)
    n0 = 0.25 / 10 ** (ebn0_db / 10)                                       # Eb = Es/4 = 0.25
    r = awgn(I + 1j * Qa, n0)
    inv = {-3: (0, 0), -1: (0, 1), 1: (1, 1), 3: (1, 0)}                   # decided level -> its 2 bits
    def decide(v):
        lv = np.clip(np.round((v * np.sqrt(10) + 3) / 2) * 2 - 3, -3, 3).astype(int)   # nearest of -3,-1,1,3
        return np.array([inv[int(l)] for l in lv])
    est = np.hstack([decide(r.real), decide(r.imag)])
    return np.mean(est != bits)

for x in (6, 8, 10, 12):
    print(f"Eb/N0 {x:2d} dB: 16-QAM sim {ber_16qam(x, 100_000):.3e}   approx 0.75 Q(sqrt(0.8 g)) = {0.75 * float(Q(np.sqrt(0.8 * 10 ** (x / 10)))):.3e}   QPSK theory {float(Q(np.sqrt(2 * 10 ** (x / 10)))):.3e}")

## 7. Misconceptions

| Wrong | Right |
|---|---|
| "SNR and Eb/N0 are the same thing" | Eb/N0 = SNR·B/Rb; they differ by bandwidth/bit-rate ratio and (per symbol) bits/symbol |
| "N0/2 per component is a typo" | One-sided PSD N0 splits equally between I and Q: variance N0/2 each |
| "QPSK has twice the BER of BPSK" | Same BER vs Eb/N0; twice the *symbol* error rate vs BPSK symbols at same Es/N0 per bit |
| "Fading just reduces SNR by a fixed amount" | Average SNR is not enough: BER slope changes from steep waterfall to 1 decade/10 dB |
| "Rayleigh = multipath" | Rayleigh = no dominant path; with LOS it is Rician |
| "Doppler shifts the carrier, a CFO tracker fixes it" | Fading changes with time (spectral broadening) — one cannot just derotate |
| "More equalisation fixes flat fading" | In flat fading the whole signal fades; equalisers help ISI, diversity/coding help fades |

## 8. Etherstack / radio context
* **Sensitivity maths:** noise floor $-174+10\log_{10}B$ dBm; + noise figure + required SNR → sensitivity. For 12.5 kHz: $-133.0$ dBm floor; NF 6 dB ⇒ −127 dBm; SNR 8.5 dB ⇒ ≈ −118.5 dBm (static, AWGN). Real LMR specs (e.g. BER 5 % at a stated dBm) are typically measured against a specific test condition — **VERIFY in the standard** (TIA-102.CAAA for P25 receiver tests; ETSI TS 102 361-2 for DMR).
* **Fading margins** in coverage planning: add the Rayleigh/Rician fade margin (+10 to +30 dB depending on reliability target) on top of static sensitivity.
* **Test equipment** emulates the channel: AWGN generator, fading simulator (Doppler, K-factor, multipath profile). The simulators here are the algorithmic core.

## 9. Exercises

### Part A — pen & paper
1. Required Eb/N0 for BPSK at BER $10^{-3}$ using $Q^{-1}(10^{-3}) = 3.09$.
2. At 9600 bit/s in 12.5 kHz, what SNR corresponds to $E_b/N_0$ = 9.6 dB? Noise floor (dBm) in 12.5 kHz with NF = 6 dB and the resulting sensitivity?
3. $f_d$ and $T_c$ for 120 km/h at 800 MHz. Number of 4800 sym/s symbols per $T_c$.
4. Compute $P(|h|^2<0.01)$ for Rayleigh and the corresponding Rician-K=6 dB statistics (variance of $|h|^2$, fraction of power in LOS).
5. Evaluate the Rayleigh BER at 20 dB and compare with AWGN (take $Q(\sqrt{200})$ as essentially 0). How many dB do you need in Rayleigh for BER $10^{-3}$?
6. Two-path channel $h=[1, 0.5]$ (delay 1 sample): peak and notch gain in dB. Where are the notches (cycles/sample)?
7. Show $E_s/N_0 = \text{SNR}_{\text{per sample}}\times$ sps for noise filling the sampling bandwidth. What is the per-sample SNR for 8 dB $E_b/N_0$ QPSK at sps = 4?
8. QPSK vs 16-QAM at BER $10^{-3}$: $\Delta E_b/N_0$?

### Part B — C++ (`code/`)
| # | Function | Concept |
|---|---|---|
| B1 | `q_function`, `bpsk_ber_awgn`, `rayleigh_bpsk_ber` | analytical curves |
| B2 | `snr_db_to_ebn0_db`, `ebn0_db_to_snr_db`, `noise_floor_dbm` | unit conversions |
| B3 | `doppler_shift_hz`, `coherence_time_s` | Doppler |
| B4 | `add_awgn`, `simulate_qpsk_ber` | Monte-Carlo BER, N0/2 per component |
| B5 | `simulate_bpsk_rayleigh_ber` | fading BER vs theory |
| B6 | `rayleigh_fading`, `rician_fading` | Clarke sum-of-sinusoids, K-factor |

### Part C — interview questions
1. Derive the BPSK BER in AWGN from the noise PDF.
2. Convert between Eb/N0 and SNR; why do we plot BER vs Eb/N0?
3. Why is BER in Rayleigh fading so much worse than in AWGN at the same average SNR?
4. What are coherence time and coherence bandwidth and how do they decide flat vs selective, fast vs slow fading?
5. How would you verify a BER simulator?

## 10. VERIFY list
* Coherence-time constant (0.423/fd) and coherence-bandwidth constant (1/(5σ)) are textbook rules of thumb — definitions vary.
* Typical urban delay spread (≈ 1–3 µs) is from memory — check for the target environment.
* Sensitivity test conditions for P25/DMR receivers: check TIA-102.CAAA / ETSI TS 102 361-2.
* P25 9600 bit/s in 12.5 kHz is quoted from Pack 05 (VERIFY there).
* The notebook cells were not executed by the author; run and compare the printed simulated vs theory columns (they should agree within a few %, except at the lowest error counts).